ROOT Version
====

Extracting a signal from two datasets.
---
Below we simulate two experiments.

Experiment1:
- has a gaussian signal
- and a falling background that goes as $exp^{-x/\lambda}$

Experiment2:
- has the same gaussian signal component
- and a background that goes as $x^n$, where $n$<0

In [1]:
import ROOT as r

In [2]:
tfsig=r.TF1("tfsig","exp(-0.5*(x-[0])*(x-[0])/[1]/[1])",25,125)
tfsig.SetParameters(75,4.5)

def experiment1():
    S_over_N = 0.08
    ndata=2700
    lam=20
    nbins=50
    xrange=(30,100)
    background = r.TF1("back1","exp(-x/[0])",xrange[0],xrange[1])
    background.SetParameter(0,lam)
    hist = r.TH1F("hexp1","Experiment1;x;frequency",nbins,xrange[0],xrange[1])
    nsig=int(ndata*S_over_N)
    nbkg=ndata-nsig
    hist.FillRandom("tfsig",nsig)
    hist.FillRandom("back1",nbkg)
    return hist

def experiment2():
    S_over_N = 0.12
    ndata=2500   
    n=-2.2
    xrange=(50,100)
    nbins=50
    background = r.TF1("back2","pow(x,[0])",xrange[0],xrange[1])
    background.SetParameter(0,n)
    hist = r.TH1F("hexp2","Experiment2;x;frequency",nbins,xrange[0],xrange[1])
    nsig=int(ndata*S_over_N)
    nbkg=ndata-nsig
    hist.FillRandom("tfsig",nsig)
    hist.FillRandom("back2",nbkg)
    return hist

Here we run the two experiments and get the results.  We will interpret these as follows:

- The experiments are independent
- They measure the same signal process
- They have different backgrounds to the signal measurement

In [3]:
tc=r.TCanvas()
tc.Divide(2,1)
h1=experiment1()
h2=experiment2()
tc.cd(1)
h1.Draw("e")
tc.cd(2)
h2.Draw("e")
tc.Draw()

Here we save the results of the experiments:

In [4]:
tf=r.TFile("experiments.root","recreate")
h1.Write()
h2.Write()
tf.Close()

And here's an example of reading them back from the TFile

Below we use DrawCopy instead of Draw, so we can close the file (which deletes the histogram from memory) without deleting the drawing.

In [6]:
tf=r.TFile("experiments.root")
h1=tf.Get("hexp1")
h2=tf.Get("hexp2")
tc.cd(1)
h1.DrawCopy("e")
tc.cd(2)
h2.DrawCopy("e")
tc.Draw()
tf.Close()

You job for this project will be to develop a simultaneous fit for the two histograms using minuit.  See this week's exercise description for more details.

### Solution

Begin by getting initial parameter guesses for the simultaneous fit by first fitting each independently

In [7]:
tf = r.TFile("experiments.root")
hexp1 = tf.Get("hexp1")
hexp2 = tf.Get("hexp2")

In [8]:
# Gaussian component (shared functional form)
gaus = r.TF1("gaus", "gaus", 20, 120)

# Exp + Gaussian for exp1
model1 = r.TF1("model1",
               "[0]*exp(-x/[1]) + "         # background part
               "[2]*exp(-0.5*((x-[3])**2)/([4]**2))", 25, 125)

# Power-law + Gaussian for exp2
model2 = r.TF1("model2",
               "[0]*pow(x, [1]) + "         # background part
               "[2]*exp(-0.5*((x-[3])**2)/([4]**2))", 50, 100)

# Initial parameter guess
model1.SetParameters(
    hexp1.GetMaximum(),            # background amplitude guess
    20,                            # lambda background slope
    int(0.05 * hexp1.Integral()),  # signal amplitude guess
    75,                            # signal mean guess
    4                              # signal sigma guess
)

# Experiment 2
model2.SetParameters(
    hexp2.GetMaximum(),            # background amplitude guess
    -2.2,                          # power-law exponent
    int(0.05 * hexp2.Integral()),  # signal amplitude guess
    75,                            # signal mean guess
    4                              # signal sigma guess
)

So with this definition:
- p0 is background amplitude
- p1 is $\lambda$ of background 
- p2 is signal amplitude
- p3 is signal mean
- p4 is signal deviation

In [9]:
c3 = r.TCanvas()
hexp1.Fit(model1)
hexp1.SetStats(0)
hexp1.Draw()
c3.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      31.6135
NDf                       =           45
Edm                       =  1.77537e-06
NCalls                    =          269
p0                        =      803.511   +/-   63.2636     
p1                        =       19.926   +/-   0.683263    
p2                        =       27.543   +/-   3.47339     
p3                        =      74.5622   +/-   0.548974    
p4                        =      4.19709   +/-   0.557192    


In [10]:
c4 = r.TCanvas()
hexp2.Fit(model2)
hexp2.Draw()
hexp2.SetStats(0)
c4.Draw()

****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      43.4008
NDf                       =           45
Edm                       =  1.93089e-06
NCalls                    =         1923
p0                        =       436425   +/-   0           
p1                        =     -2.16741   +/-   0           
p2                        =      23.7673   +/-   0           
p3                        =       74.786   +/-   0           
p4                        =      5.06667   +/-   0           


Warning in <Fit>: Abnormal termination of minimization.


So now we have some initial parameters

In [11]:
tf = r.TFile("experiments.root")
hexp1 = tf.Get("hexp1")
hexp2 = tf.Get("hexp2")

In [12]:
# signal is shared: mean=[0], sigma=[1]
gaus = r.TF1("gaus", "[2]*exp(-0.5*((x-[0])**2)/([1]**2))", 0, 200)

# Define combined models for each experiment
# exp1: Gaussian signal + exponential background
model1_sim = r.TF1("model1_sim", "[0]*exp(-x/[1]) + [2]*exp(-0.5*((x-[3])**2)/([4]**2))", 0, 200)
# params: [0]=A1, [1]=lambda, [2]=S1, [3]=mean, [4]=sigma

# exp2: Gaussian signal + power-law background  
model2_sim = r.TF1("model2_sim", "[0]*pow(x,[1]) + [2]*exp(-0.5*((x-[3])**2)/([4]**2))", 0, 200)
# params: [0]=A2, [1]=n, [2]=S2, [3]=mean, [4]=sigma

- 0: A_exp1
- 1: lambda1
- 2: S1 amplitude
- 3: mean (shared)
- 4: sigma (shared)
- 5: A_pow2
- 6: n2
- 7: S2 amplitude

In [13]:
# custom objective function that MINUIT minimizes

def chi2_fun(placeholder, p):
    # "p" is input list of parameters, MINUIT will find best ones
    # "placeholder" is so that this psuedofunction can be treated like a TF1 later 
    
    chi2 = 0.0
    
    # Set parameters: exp1 uses p[0-4], exp2 uses p[5-7] and shared p[3,4]
    model1_sim.SetParameters(p[0], p[1], p[2], p[3], p[4])
    model2_sim.SetParameters(p[5], p[6], p[7], p[3], p[4])
    
    # for exp1
    for i in range(1, hexp1.GetNbinsX()+1):
        x_bin = hexp1.GetBinCenter(i)
        if x_bin < 25 or x_bin > 125:  # Respect fit range
            continue
        obs = hexp1.GetBinContent(i)
        if obs <= 0: 
            continue
        exp_val = model1_sim.Eval(x_bin)
        err = hexp1.GetBinError(i)
        chi2 += ((obs - exp_val)/err)**2
    
    # for exp2
    for i in range(1, hexp2.GetNbinsX()+1):
        x_bin = hexp2.GetBinCenter(i)
        if x_bin < 50 or x_bin > 100:  # Respect fit range
            continue
        obs = hexp2.GetBinContent(i)
        if obs <= 0:
            continue
        exp_val = model2_sim.Eval(x_bin)
        err = hexp2.GetBinError(i)
        chi2 += ((obs - exp_val)/err)**2
    
    return chi2

In [14]:
# turn into a TF1
chi2_total = r.TF1("chi2_total", chi2_fun, 0, 1, 8)

In [15]:
# initial guesses
chi2_total.SetParameters(
    800, 20, 28,   # exp1: bkg_norm, lambda, S1
    74.6, 4.2,     # mean, sigma (shared)
    250, -2.2, 25  # exp2: bkg_norm, power, S2
)

In [16]:
# Fit the dummy histogram from exp2, just a trick to make the MINUIT minimizer run
hexp2.Fit(chi2_total, "SN")
# ignore the chi2 here

****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      4278.35
NDf                       =           42
Edm                       =      466.935
NCalls                    =         2129
p0                        =      1028.67   +/-   3.74509     
p1                        =      17.9092   +/-   0.0234096   
p2                        =      27.2359   +/-   0.111771    
p3                        =      74.4093   +/-   0.0436739   
p4                        =      4.70073   +/-   0.0410383   
p5                        =      85125.2   +/-   349.837     
p6                        =     -1.77605   +/-   0.00106092  
p7                        =      20.7176   +/-   0.223009    


Warning in <Fit>: Abnormal termination of minimization.


Now that the parameters that minimize the combined chi2 are found, manually calculate the chi2 for the model functions with these parameters

In [17]:
from ROOT import TMath

In [18]:
chi2_manual = 0
ndf = 0

for data, model, low, high, offset in [(hexp1, model1_sim, 25, 125, 0), (hexp2, model2_sim, 50, 100, 5)]:
    for i in range(1, data.GetNbinsX()+1):
        x = data.GetBinCenter(i)
        if x < low or x > high: 
            continue
        obs = data.GetBinContent(i)
        err = data.GetBinError(i)
        exp_val = model.Eval(x)
        chi2_manual += ((obs - exp_val)/err)**2
        ndf += 1

ndf = ndf - 8  # number of free parameters
print("Chi2 =", chi2_manual)

prob = TMath.Prob(chi2_manual, ndf)
print("p-value =", prob)

Chi2 = 96.39026578354685
p-value = 0.3565761593962682


In [19]:
print("Simultaneous Fit Results")
for i in range(8):
    print(f"p{i} = {chi2_total.GetParameter(i):.4f} ± {chi2_total.GetParError(i):.4f}")

mu, sigma = chi2_total.GetParameter(3), chi2_total.GetParameter(4)
emu, esigma = chi2_total.GetParError(3), chi2_total.GetParError(4)

print("\nchi2 =", chi2_manual)
print("prob =", prob)

print("\nSignal Parameters")
print("mean =", mu, "+/-", emu)
print("sigma =", sigma, "+/-", esigma)

Simultaneous Fit Results
p0 = 1028.6708 ± 3.7451
p1 = 17.9092 ± 0.0234
p2 = 27.2359 ± 0.1118
p3 = 74.4093 ± 0.0437
p4 = 4.7007 ± 0.0410
p5 = 85125.1611 ± 349.8365
p6 = -1.7760 ± 0.0011
p7 = 20.7176 ± 0.2230

chi2 = 96.39026578354685
prob = 0.3565761593962682

Signal Parameters
mean = 74.40932502239635 +/- 0.04367385388676266
sigma = 4.700730905330066 +/- 0.041038328731545504


In [20]:
pars = [chi2_total.GetParameter(i) for i in range(8)]
errs = [chi2_total.GetParError(i) for i in range(8)]

c = r.TCanvas()
c.Divide(2,1)

# Experiment 1
c.cd(1)
hexp1.Draw("hist e")
hexp1.SetStats(0)

model1_sim.SetParameters(pars[0], pars[1], pars[2], mu, sigma)
model1_sim.SetLineColor(r.kRed)
model1_sim.Draw("same")

l = r.TLatex()
l.SetNDC()
l.SetTextSize(0.04)
l.DrawLatex(0.3, 0.85, f"mean = {mu:.2f} ± {emu:.2f}")
l.DrawLatex(0.3, 0.80, f"sigma = {sigma:.2f} ± {esigma:.2f}")
l.DrawLatex(0.3, 0.75, f"chi2 = {chi2_manual:.3f}")
l.DrawLatex(0.3, 0.70, f"chi2 prob = {prob:.3f}")

# Experiment 2
c.cd(2)
hexp2.Draw("hist e")
hexp2.SetStats(0)


model2_sim.SetParameters(pars[5], pars[6], pars[7], mu, sigma)
model2_sim.SetLineColor(r.kRed)
model2_sim.Draw("same")

l2 = r.TLatex()
l2.SetNDC()
l2.SetTextSize(0.04)
l2.DrawLatex(0.3, 0.85, f"mean = {mu:.2f} ± {emu:.2f}")
l2.DrawLatex(0.3, 0.80, f"sigma = {sigma:.2f} ± {esigma:.2f}")
l2.DrawLatex(0.3, 0.75, f"chi2 = {chi2_manual:.3f}")
l2.DrawLatex(0.3, 0.70, f"chi2 prob = {prob:.3f}")

#c.SaveAs("ex2.pdf")
c.Draw()


### An alternative method using ROOT Minuit2 minimizer more directly 

In [5]:
import ROOT as r
import numpy as np

In [6]:
# Open the file with the data
tf = r.TFile("experiments.root")
hexp1 = tf.Get("hexp1")
hexp2 = tf.Get("hexp2")

In [7]:
# Get histogram ranges and bin information
xmin1, xmax1 = hexp1.GetXaxis().GetXmin(), hexp1.GetXaxis().GetXmax()
xmin2, xmax2 = hexp2.GetXaxis().GetXmin(), hexp2.GetXaxis().GetXmax()

# Define the model functions
# Experiment 1: Gaussian signal + exponential background
model1 = r.TF1("model1", "[0]*exp(-0.5*(x-[1])*(x-[1])/[2]/[2]) + [3]*exp(-x/[4])", xmin1, xmax1)
model1.SetParNames("N1_sig", "mean", "sigma", "N1_bkg", "lambda")

# Experiment 2: Gaussian signal + power-law background  
model2 = r.TF1("model2", "[0]*exp(-0.5*(x-[1])*(x-[1])/[2]/[2]) + [3]*pow(x,[4])", xmin2, xmax2)
model2.SetParNames("N2_sig", "mean", "sigma", "N2_bkg", "n")

# Set initial parameter values
# Signal parameters (common)
mean_init = 75.0
sigma_init = 4.5

# Experiment 1 specific
N1_sig_init = 200.0
N1_bkg_init = 2500.0
lambda_init = 20.0

# Experiment 2 specific
N2_sig_init = 300.0
N2_bkg_init = 2200.0
n_init = -2.2

model1.SetParameters(N1_sig_init, mean_init, sigma_init, N1_bkg_init, lambda_init)
model2.SetParameters(N2_sig_init, mean_init, sigma_init, N2_bkg_init, n_init)

# Set parameter limits
model1.SetParLimits(2, 1.0, 10.0)  # sigma > 0
model1.SetParLimits(4, 5.0, 50.0)  # lambda > 0
model2.SetParLimits(2, 1.0, 10.0)  # sigma > 0
model2.SetParLimits(4, -5.0, -0.5)  # n < 0

# Create a global chi2 function
class GlobalChi2:
    def __init__(self, hist1, hist2, func1, func2):
        self.hexp1 = hist1
        self.hexp2 = hist2
        self.f1 = func1
        self.f2 = func2
        self.nbins1 = hist1.GetNbinsX()
        self.nbins2 = hist2.GetNbinsX()
        
    def __call__(self, par):
        # Set parameters for experiment 1
        # par[0]=N1_sig, par[1]=mean, par[2]=sigma, par[3]=N1_bkg, par[4]=lambda
        self.f1.SetParameter(0, par[0])
        self.f1.SetParameter(1, par[1])
        self.f1.SetParameter(2, par[2])
        self.f1.SetParameter(3, par[3])
        self.f1.SetParameter(4, par[4])
        
        # Set parameters for experiment 2
        # par[5]=N2_sig, par[1]=mean (shared), par[2]=sigma (shared), 
        # par[6]=N2_bkg, par[7]=n
        self.f2.SetParameter(0, par[5])
        self.f2.SetParameter(1, par[1])
        self.f2.SetParameter(2, par[2])
        self.f2.SetParameter(3, par[6])
        self.f2.SetParameter(4, par[7])
        
        # Calculate chi2 for experiment 1
        chi2_1 = 0.0
        for i in range(1, self.nbins1 + 1):
            x = self.hexp1.GetBinCenter(i)
            y = self.hexp1.GetBinContent(i)
            err = self.hexp1.GetBinError(i)
            if err > 0:
                chi2_1 += ((y - self.f1.Eval(x)) / err) ** 2
        
        # Calculate chi2 for experiment 2
        chi2_2 = 0.0
        for i in range(1, self.nbins2 + 1):
            x = self.hexp2.GetBinCenter(i)
            y = self.hexp2.GetBinContent(i)
            err = self.hexp2.GetBinError(i)
            if err > 0:
                chi2_2 += ((y - self.f2.Eval(x)) / err) ** 2
        
        return chi2_1 + chi2_2

# Setup the minimizer
global_chi2 = GlobalChi2(h1, h2, model1, model2)

minimizer = r.Math.Factory.CreateMinimizer("Minuit2", "Migrad")
minimizer.SetMaxFunctionCalls(100000)
minimizer.SetTolerance(0.01)
minimizer.SetPrintLevel(1)

# Setup parameters: N1_sig, mean, sigma, N1_bkg, lambda, N2_sig, N2_bkg, n
functor = r.Math.Functor(global_chi2, 8)
minimizer.SetFunction(functor)

minimizer.SetVariable(0, "N1_sig", N1_sig_init, 10.0)
minimizer.SetVariable(1, "mean", mean_init, 0.1)
minimizer.SetVariable(2, "sigma", sigma_init, 0.1)
minimizer.SetVariable(3, "N1_bkg", N1_bkg_init, 10.0)
minimizer.SetVariable(4, "lambda", lambda_init, 1.0)
minimizer.SetVariable(5, "N2_sig", N2_sig_init, 10.0)
minimizer.SetVariable(6, "N2_bkg", N2_bkg_init, 10.0)
minimizer.SetVariable(7, "n", n_init, 0.1)

# Set limits
minimizer.SetVariableLimits(2, 1.0, 10.0)   # sigma
minimizer.SetVariableLimits(4, 5.0, 50.0)   # lambda
minimizer.SetVariableLimits(7, -5.0, -0.5)  # n

# Perform the fit
minimizer.Minimize()

# Get results
xs = minimizer.X()
errors = minimizer.Errors()

# Update function parameters with fit results
model1.SetParameter(0, xs[0])
model1.SetParameter(1, xs[1])
model1.SetParameter(2, xs[2])
model1.SetParameter(3, xs[3])
model1.SetParameter(4, xs[4])

model2.SetParameter(0, xs[5])
model2.SetParameter(1, xs[1])
model2.SetParameter(2, xs[2])
model2.SetParameter(3, xs[6])
model2.SetParameter(4, xs[7])

Minuit2Minimizer: Minimize with max-calls 100000 convergence for edm < 0.01 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 76.1675742377073561
Edm   = 1.00343782601853413e-06
Nfcn  = 862
N1_sig	  = 26.4284	 +/-  3.24655
mean	  = 74.6042	 +/-  0.451361
sigma	  = 4.59603	 +/-  0.443836	(limited)
N1_bkg	  = 819.826	 +/-  64.2382
lambda	  = 19.7234	 +/-  0.664572	(limited)
N2_sig	  = 24.2821	 +/-  3.22076
N2_bkg	  = 386343	 +/-  214157
n	  = -2.13612	 +/-  0.132912	(limited)


In [12]:
print("\nCOMMON SIGNAL PARAMETERS:")
print(f"  Mean  = {xs[1]:.3f} ± {errors[1]:.3f}")
print(f"  Sigma = {xs[2]:.3f} ± {errors[2]:.3f}")

print("\nEXPERIMENT 1:")
print(f"  N_signal     = {xs[0]:.1f} ± {errors[0]:.1f}")
print(f"  N_background = {xs[3]:.1f} ± {errors[3]:.1f}")
print(f"  lambda       = {xs[4]:.3f} ± {errors[4]:.3f}")

print("\nEXPERIMENT 2:")
print(f"  N_signal     = {xs[5]:.1f} ± {errors[5]:.1f}")
print(f"  N_background = {xs[6]:.1f} ± {errors[6]:.1f}")
print(f"  n            = {xs[7]:.3f} ± {errors[7]:.3f}")

print(f"\nMinimum chi2 = {minimizer.MinValue():.2f}")
ndf = hexp1.GetNbinsX() + hexp2.GetNbinsX() - 8
print(f"NDF          = {ndf}")
print(f"chi2/NDF     = {minimizer.MinValue()/ndf:.2f}")
print(f"Prob(chi2)   = {r.TMath.Prob(minimizer.MinValue(), ndf):.4f}")


COMMON SIGNAL PARAMETERS:
  Mean  = 74.604 ± 0.451
  Sigma = 4.596 ± 0.444

EXPERIMENT 1:
  N_signal     = 26.4 ± 3.2
  N_background = 819.8 ± 64.2
  lambda       = 19.723 ± 0.665

EXPERIMENT 2:
  N_signal     = 24.3 ± 3.2
  N_background = 386343.5 ± 214156.7
  n            = -2.136 ± 0.133

Minimum chi2 = 76.17
NDF          = 92
chi2/NDF     = 0.83
Prob(chi2)   = 0.8833


In [15]:
# Create components for plotting
bkg1 = r.TF1("bkg1", "[0]*exp(-x/[1])", xmin1, xmax1)
bkg1.SetParameters(xs[3], xs[4])
bkg1.SetLineColor(r.kBlue)
bkg1.SetLineStyle(2)

bkg2 = r.TF1("bkg2", "[0]*pow(x,[1])", xmin2, xmax2)
bkg2.SetParameters(xs[6], xs[7])
bkg2.SetLineColor(r.kBlue)
bkg2.SetLineStyle(2)

model1.SetLineColor(r.kRed)
model2.SetLineColor(r.kRed)

# Draw results
c1 = r.TCanvas()
c1.Divide(2, 1)

c1.cd(1)
hexp1.Draw("hist e")
hexp1.SetStats(0)
model1.Draw("same")
bkg1.Draw("same")
legend1 = r.TLegend(0.6, 0.65, 0.88, 0.88)
legend1.AddEntry(h1, "Data", "lep")
legend1.AddEntry(model1, "Total fit", "l")
legend1.AddEntry(bkg1, "Background", "l")
legend1.Draw()

# Add text with fit results
text1 = r.TLatex()
text1.SetNDC()
text1.SetTextSize(0.035)
text1.DrawLatex(0.3, 0.85, f"#mu = {xs[1]:.2f} #pm {errors[1]:.2f}")
text1.DrawLatex(0.3, 0.80, f"#sigma = {xs[2]:.2f} #pm {errors[2]:.2f}")
text1.DrawLatex(0.3, 0.75, f"Prob(#chi^{{2}}) = {r.TMath.Prob(minimizer.MinValue(), ndf):.3f}")

c1.cd(2)
hexp2.Draw("hist e")
hexp2.SetStats(0)
model2.Draw("same")
bkg2.Draw("same")
legend2 = r.TLegend(0.6, 0.65, 0.88, 0.88)
legend2.AddEntry(h2, "Data", "lep")
legend2.AddEntry(model2, "Total fit", "l")
legend2.AddEntry(bkg2, "Background", "l")
legend2.Draw()

# Add text with fit results
text2 = r.TLatex()
text2.SetNDC()
text2.SetTextSize(0.035)
text2.DrawLatex(0.3, 0.85, f"#mu = {xs[1]:.2f} #pm {errors[1]:.2f}")
text2.DrawLatex(0.3, 0.80, f"#sigma = {xs[2]:.2f} #pm {errors[2]:.2f}")
text2.DrawLatex(0.3, 0.75, f"Prob(#chi^{{2}}) = {r.TMath.Prob(minimizer.MinValue(), ndf):.3f}")

c1.Draw()
c1.SaveAs("ex2.pdf")

Info in <TCanvas::Print>: pdf file ex2.pdf has been created


In [63]:
import ROOT as r
import array

# Your data and models
tf = r.TFile("experiments.root")
hexp1 = tf.Get("hexp1")
hexp2 = tf.Get("hexp2")

# Define models
model1_sim = r.TF1("model1_sim", 
                   "[0]*exp(-x/[1]) + [2]*exp(-0.5*((x-[3])**2)/([4]**2))", 
                   25, 125)

model2_sim = r.TF1("model2_sim", 
                   "[0]*pow(x,[1]) + [2]*exp(-0.5*((x-[3])**2)/([4]**2))", 
                   50, 100)

# Global chi² function that MINUIT will call
def chi2_fcn(npar, gin, f, par, iflag):
    """
    MINUIT's function signature
    """
    chi2 = 0.0
    
    # Set parameters from MINUIT
    model1_sim.SetParameters(par[0], par[1], par[2], par[3], par[4])
    model2_sim.SetParameters(par[5], par[6], par[7], par[3], par[4])
    
    # Chi² for exp1
    for i in range(1, hexp1.GetNbinsX()+1):
        x_bin = hexp1.GetBinCenter(i)
        if x_bin < 25 or x_bin > 125:
            continue
        obs = hexp1.GetBinContent(i)
        if obs <= 0: 
            continue
        exp_val = model1_sim.Eval(x_bin)
        err = hexp1.GetBinError(i)
        chi2 += ((obs - exp_val)/err)**2
    
    # Chi² for exp2
    for i in range(1, hexp2.GetNbinsX()+1):
        x_bin = hexp2.GetBinCenter(i)
        if x_bin < 50 or x_bin > 100:
            continue
        obs = hexp2.GetBinContent(i)
        if obs <= 0:
            continue
        exp_val = model2_sim.Eval(x_bin)
        err = hexp2.GetBinError(i)
        chi2 += ((obs - exp_val)/err)**2
    
    # Return chi² value
    return chi2

# Create MINUIT instance
minuit = r.TMinuit(8)  # 8 parameters
minuit.SetFCN(chi2_fcn)
minuit.SetPrintLevel(1)  # Control verbosity

# Set up parameters
param_names = ["A1", "lambda", "S1", "mu", "sigma", "A2", "n", "S2"]
initial_values = [800, 20, 28, 74.6, 4.2, 250, -2.2, 25]
step_sizes = [10, 0.1, 1, 0.1, 0.1, 10, 0.1, 1]

for i, (name, val, step) in enumerate(zip(param_names, initial_values, step_sizes)):
    minuit.DefineParameter(i, name, val, step, 0, 0)

# Set parameter limits
minuit.SetErrorDef(1)  # For chi² fits

# Use Command strings (simpler approach)
print("\n=== Setting up MINUIT ===")
minuit.Command("SET LIM 2 0.1 100")    # lambda > 0
minuit.Command("SET LIM 5 0.1 50")     # sigma > 0  
minuit.Command("SET LIM 7 -10 -0.1")   # n < 0

# Run MIGRAD
print("\n=== Running MINUIT ===")
minuit.Command("MIGRAD 5000 0.1")

# Get results using arrays
best_params = []
errors = []
for i in range(8):
    val = array.array('d', [0])
    err = array.array('d', [0])
    minuit.GetParameter(i, val, err)
    best_params.append(val[0])
    errors.append(err[0])

print("\n=== Best-Fit Parameters ===")
for i, name in enumerate(param_names):
    print(f"{name:8s} = {best_params[i]:10.4f} ± {errors[i]:8.4f}")

# Calculate chi² manually with best-fit parameters
model1_sim.SetParameters(*best_params[:5])
model2_sim.SetParameters(best_params[5], best_params[6], best_params[7], 
                         best_params[3], best_params[4])

chi2_min = 0.0
ndf = 0

# Chi² for exp1
for i in range(1, hexp1.GetNbinsX()+1):
    x = hexp1.GetBinCenter(i)
    if x < 25 or x > 125:
        continue
    obs = hexp1.GetBinContent(i)
    if obs <= 0:
        continue
    err = hexp1.GetBinError(i)
    exp_val = model1_sim.Eval(x)
    chi2_min += ((obs - exp_val)/err)**2
    ndf += 1

# Chi² for exp2
for i in range(1, hexp2.GetNbinsX()+1):
    x = hexp2.GetBinCenter(i)
    if x < 50 or x > 100:
        continue
    obs = hexp2.GetBinContent(i)
    if obs <= 0:
        continue
    err = hexp2.GetBinError(i)
    exp_val = model2_sim.Eval(x)
    chi2_min += ((obs - exp_val)/err)**2
    ndf += 1

ndf -= 8  # Subtract parameters

print(f"\n=== Goodness of Fit ===")
print(f"Chi²     = {chi2_min:.2f}")
print(f"NDF      = {ndf}")
print(f"Chi²/NDF = {chi2_min/ndf:.3f}")
print(f"p-value  = {r.TMath.Prob(chi2_min, ndf):.4f}")


=== Setting up MINUIT ===

=== Running MINUIT ===

=== Best-Fit Parameters ===
A1       =   800.0300 ±      inf
lambda   =    20.0000 ±      nan
S1       =    28.0000 ±      inf
mu       =    74.6000 ±      inf
sigma    =     4.2000 ±      nan
A2       =   250.0000 ±      inf
n        =    -2.2000 ±      nan
S2       =    25.0000 ±      inf

=== Goodness of Fit ===
Chi²     = 2083.75
NDF      = 92
Chi²/NDF = 22.649
p-value  = 0.0000
 **********
 **    1 **SET PRINT           1
 **********
 PARAMETER DEFINITIONS:
    NO.   NAME         VALUE      STEP SIZE      LIMITS
     1 A1           8.00000e+02  1.00000e+01     no limits
     2 lambda       2.00000e+01  1.00000e-01     no limits
     3 S1           2.80000e+01  1.00000e+00     no limits
     4 mu           7.46000e+01  1.00000e-01     no limits
     5 sigma        4.20000e+00  1.00000e-01     no limits
     6 A2           2.50000e+02  1.00000e+01     no limits
     7 n           -2.20000e+00  1.00000e-01     no limits
     8 S2   